# Interacting with Uniswap using Universal Router
In last week, we interacted with Uniswap using their GUI. This week, we will interact with Uniswap using code.
We understand how to build transactions and approve the right smart contracts to use our funds to perform these swaps. 

<img src="./figs/uniswap_f1.png" width="600"/>

Then, we execute an atomic swap via Code.

### ERC-2612 vs Permit2
- `ERC-2612` is a token standard (technical specification)
- `Permit2` is a specific smart contract (infrastructure) developed by Uniswap.
- Both aim to solve the same problem: the cumbersome and expensive Token approval mechanism.

### ERC-2612: Gasless Approval via Signatures
To understand ERC-2612, we first look at how the **traditional ERC-20** works.

#### Traditional Pain Point (ERC-20)
If you want to deposit USDT into Uniswap for trading, you must complete **two separate transactions**:
1. **Approve**: Send a transaction to the USDT contract to authorize: “I allow Uniswap to spend my 100 USDT.”
   This **requires Gas** and waits for on-chain confirmation.
2. **Transfer/Execute**: Send another transaction to Uniswap to start trading. Only then can Uniswap transfer your tokens.
   This **costs Gas again**.

**Drawback**: Poor user experience (two waits) and high Gas fees.

#### ERC-2612: Signature-Based Approval
ERC-2612 extends ERC-20 by adding a `permit` function.
1. **Sign (off-chain)**: Instead of sending an on-chain transaction, you simply **sign a message** in your wallet (free, off-chain).
   The signature states: “I authorize Uniswap to spend 100 USDT until tomorrow.”
2. **Execute**: You send the signature to Uniswap.
   During the swap, Uniswap uses your signature to call the token’s `permit` function (instant approval), then completes the transfer.

**Advantage**: Two steps merged into one, saves Gas, and improves UX.

In summary:
It allows tokens to support **approval via signature**, rather than requiring an on-chain transaction.


## Populate all relavant variables

In [96]:
# Import web3 library
from web3 import Web3

# Setup variables
infura_key = '2e306bdddc7843108fe30334b2dfcfb2'  # Replace with your Infura Project ID or set in .env
wallet_public_address = '0xECfa7eCDAd56aeb78c4B5319b9446f34D2F68969'  # Replace with your wallet address or set in .env
wallet_private_key = '7be192a533a03484ba349c66e3c76566167115c577f282a48524177d88a8a9df'  # Replace with your private key or set in .env


w3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))


print(w3.__dict__)
print(w3.eth.get_block_number())

wallet_public_address = Web3.to_checksum_address(wallet_public_address)

# Define token addresses (USTUSD, USTETH), where USTUSD is the wrapped Ether token
# WETH_token_address = Web3.to_checksum_address('0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14')
USTUSD_token_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
USTETH_token_address = Web3.to_checksum_address('0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057')


# USTUSD token contract ABI (ERC-20 standard): https://sepolia.etherscan.io/address/0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14#code 
abi_USTUSD = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"allowance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"},{"internalType":"uint256","name":"balance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientBalance","type":"error"},{"inputs":[{"internalType":"address","name":"approver","type":"address"}],"name":"ERC20InvalidApprover","type":"error"},{"inputs":[{"internalType":"address","name":"receiver","type":"address"}],"name":"ERC20InvalidReceiver","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"ERC20InvalidSender","type":"error"},{"inputs":[{"internalType":"address","name":"spender","type":"address"}],"name":"ERC20InvalidSpender","type":"error"},{"inputs":[{"internalType":"address","name":"owner","type":"address"}],"name":"OwnableInvalidOwner","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"OwnableUnauthorizedAccount","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"previousOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Transfer","type":"event"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"approve","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"balanceOf","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"mint","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"name","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"renounceOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"symbol","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"totalSupply","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transfer","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transferFrom","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"newOwner","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

# USTETH token contract ABI (ERC-20 standard): https://sepolia.etherscan.io/address/0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057#code
abi_USTETH = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"allowance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"},{"internalType":"uint256","name":"balance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientBalance","type":"error"},{"inputs":[{"internalType":"address","name":"approver","type":"address"}],"name":"ERC20InvalidApprover","type":"error"},{"inputs":[{"internalType":"address","name":"receiver","type":"address"}],"name":"ERC20InvalidReceiver","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"ERC20InvalidSender","type":"error"},{"inputs":[{"internalType":"address","name":"spender","type":"address"}],"name":"ERC20InvalidSpender","type":"error"},{"inputs":[{"internalType":"address","name":"owner","type":"address"}],"name":"OwnableInvalidOwner","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"OwnableUnauthorizedAccount","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"previousOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Transfer","type":"event"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"approve","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"balanceOf","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"mint","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"name","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"renounceOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"symbol","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"totalSupply","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transfer","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transferFrom","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"newOwner","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

USTUSD_contract = w3.eth.contract(address=USTUSD_token_address, abi=abi_USTUSD)
USTETH_contract = w3.eth.contract(address=USTETH_token_address, abi=abi_USTETH)

print(USTUSD_contract.functions.name().call())
print(USTETH_contract.functions.name().call())

{'manager': <web3.manager.RequestManager object at 0x00000201C2990040>, 'codec': <eth_abi.codec.ABICodec object at 0x00000201C3BD6950>, 'eth': <web3.eth.eth.Eth object at 0x00000201C3BD4460>, 'net': <web3.net.Net object at 0x00000201C3BD5870>, 'geth': <web3.geth.Geth object at 0x00000201C3BD5BD0>, 'tracing': <web3.tracing.Tracing object at 0x00000201C3DDC220>, 'testing': <web3.testing.Testing object at 0x00000201C3DDC8E0>, '_ens': <web3._utils.empty.Empty object at 0x00000201C16ED1B0>}
10361154
HKUSTGZ USD
HKUSTGZ ETH


### Universal Router
- Help us to route our swaps (e.g., USDT -> USDC -> ETH) in a single transaction with the best price.
- It is the intermediary that orchestrates how funds flow, handling multi-hop trades, slippage protection, deadline checks, etc.
- It supports both ERC-2612 and Permit2, allowing users to choose their preferred approval method.


In [98]:
# Define Universal Router address and ABI: https://sepolia.etherscan.io/address/0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD#code
universal_router_address = Web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')

abi_universal_router = '[{"inputs":[{"components":[{"internalType":"address","name":"permit2","type":"address"},{"internalType":"address","name":"USTUSD9","type":"address"},{"internalType":"address","name":"seaportV1_5","type":"address"},{"internalType":"address","name":"seaportV1_4","type":"address"},{"internalType":"address","name":"openseaConduit","type":"address"},{"internalType":"address","name":"nftxZap","type":"address"},{"internalType":"address","name":"x2y2","type":"address"},{"internalType":"address","name":"foundation","type":"address"},{"internalType":"address","name":"sudoswap","type":"address"},{"internalType":"address","name":"elementMarket","type":"address"},{"internalType":"address","name":"nft20Zap","type":"address"},{"internalType":"address","name":"cryptopunks","type":"address"},{"internalType":"address","name":"looksRareV2","type":"address"},{"internalType":"address","name":"routerRewardsDistributor","type":"address"},{"internalType":"address","name":"looksRareRewardsDistributor","type":"address"},{"internalType":"address","name":"looksRareToken","type":"address"},{"internalType":"address","name":"v2Factory","type":"address"},{"internalType":"address","name":"v3Factory","type":"address"},{"internalType":"bytes32","name":"pairInitCodeHash","type":"bytes32"},{"internalType":"bytes32","name":"poolInitCodeHash","type":"bytes32"}],"internalType":"struct RouterParameters","name":"params","type":"tuple"}],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[],"name":"BalanceTooLow","type":"error"},{"inputs":[],"name":"BuyPunkFailed","type":"error"},{"inputs":[],"name":"ContractLocked","type":"error"},{"inputs":[],"name":"ETHNotAccepted","type":"error"},{"inputs":[{"internalType":"uint256","name":"commandIndex","type":"uint256"},{"internalType":"bytes","name":"message","type":"bytes"}],"name":"ExecutionFailed","type":"error"},{"inputs":[],"name":"FromAddressIsNotOwner","type":"error"},{"inputs":[],"name":"InsufficientETH","type":"error"},{"inputs":[],"name":"InsufficientToken","type":"error"},{"inputs":[],"name":"InvalidBips","type":"error"},{"inputs":[{"internalType":"uint256","name":"commandType","type":"uint256"}],"name":"InvalidCommandType","type":"error"},{"inputs":[],"name":"InvalidOwnerERC1155","type":"error"},{"inputs":[],"name":"InvalidOwnerERC721","type":"error"},{"inputs":[],"name":"InvalidPath","type":"error"},{"inputs":[],"name":"InvalidReserves","type":"error"},{"inputs":[],"name":"InvalidSpender","type":"error"},{"inputs":[],"name":"LengthMismatch","type":"error"},{"inputs":[],"name":"SliceOutOfBounds","type":"error"},{"inputs":[],"name":"TransactionDeadlinePassed","type":"error"},{"inputs":[],"name":"UnableToClaim","type":"error"},{"inputs":[],"name":"UnsafeCast","type":"error"},{"inputs":[],"name":"V2InvalidPath","type":"error"},{"inputs":[],"name":"V2TooLittleReceived","type":"error"},{"inputs":[],"name":"V2TooMuchRequested","type":"error"},{"inputs":[],"name":"V3InvalidAmountOut","type":"error"},{"inputs":[],"name":"V3InvalidCaller","type":"error"},{"inputs":[],"name":"V3InvalidSwap","type":"error"},{"inputs":[],"name":"V3TooLittleReceived","type":"error"},{"inputs":[],"name":"V3TooMuchRequested","type":"error"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"amount","type":"uint256"}],"name":"RewardsSent","type":"event"},{"inputs":[{"internalType":"bytes","name":"looksRareClaim","type":"bytes"}],"name":"collectRewards","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"bytes","name":"commands","type":"bytes"},{"internalType":"bytes[]","name":"inputs","type":"bytes[]"}],"name":"execute","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"bytes","name":"commands","type":"bytes"},{"internalType":"bytes[]","name":"inputs","type":"bytes[]"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"name":"execute","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256[]","name":"","type":"uint256[]"},{"internalType":"uint256[]","name":"","type":"uint256[]"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC1155BatchReceived","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC1155Received","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC721Received","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"bytes4","name":"interfaceId","type":"bytes4"}],"name":"supportsInterface","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"int256","name":"amount0Delta","type":"int256"},{"internalType":"int256","name":"amount1Delta","type":"int256"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"uniswapV3SwapCallback","outputs":[],"stateMutability":"nonpayable","type":"function"},{"stateMutability":"payable","type":"receive"}]'

# Define Permit2 address and ABI: https://sepolia.etherscan.io/address/0x000000000022D473030F116dDEE9F6B43aC78BA3#code
permit2_address = w3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
abi_permit2 = '[{"inputs":[{"internalType":"uint256","name":"deadline","type":"uint256"}],"name":"AllowanceExpired","type":"error"},{"inputs":[],"name":"ExcessiveInvalidation","type":"error"},{"inputs":[{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"uint256","name":"maxAmount","type":"uint256"}],"name":"InvalidAmount","type":"error"},{"inputs":[],"name":"InvalidContractSignature","type":"error"},{"inputs":[],"name":"InvalidNonce","type":"error"},{"inputs":[],"name":"InvalidSignature","type":"error"},{"inputs":[],"name":"InvalidSignatureLength","type":"error"},{"inputs":[],"name":"InvalidSigner","type":"error"},{"inputs":[],"name":"LengthMismatch","type":"error"},{"inputs":[{"internalType":"uint256","name":"signatureDeadline","type":"uint256"}],"name":"SignatureExpired","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint160","name":"amount","type":"uint160"},{"indexed":false,"internalType":"uint48","name":"expiration","type":"uint48"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"address","name":"token","type":"address"},{"indexed":false,"internalType":"address","name":"spender","type":"address"}],"name":"Lockdown","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint48","name":"newNonce","type":"uint48"},{"indexed":false,"internalType":"uint48","name":"oldNonce","type":"uint48"}],"name":"NonceInvalidation","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint160","name":"amount","type":"uint160"},{"indexed":false,"internalType":"uint48","name":"expiration","type":"uint48"},{"indexed":false,"internalType":"uint48","name":"nonce","type":"uint48"}],"name":"Permit","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"uint256","name":"word","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"mask","type":"uint256"}],"name":"UnorderedNonceInvalidation","type":"event"},{"inputs":[],"name":"DOMAIN_SEPARATOR","outputs":[{"internalType":"bytes32","name":"","type":"bytes32"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"}],"name":"approve","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint48","name":"newNonce","type":"uint48"}],"name":"invalidateNonces","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"wordPos","type":"uint256"},{"internalType":"uint256","name":"mask","type":"uint256"}],"name":"invalidateUnorderedNonces","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"internalType":"struct IAllowanceTransfer.TokenSpenderPair[]","name":"approvals","type":"tuple[]"}],"name":"lockdown","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"}],"name":"nonceBitmap","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"internalType":"struct IAllowanceTransfer.PermitDetails[]","name":"details","type":"tuple[]"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"sigDeadline","type":"uint256"}],"internalType":"struct IAllowanceTransfer.PermitBatch","name":"permitBatch","type":"tuple"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permit","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"internalType":"struct IAllowanceTransfer.PermitDetails","name":"details","type":"tuple"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"sigDeadline","type":"uint256"}],"internalType":"struct IAllowanceTransfer.PermitSingle","name":"permitSingle","type":"tuple"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permit","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions","name":"permitted","type":"tuple"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails","name":"transferDetails","type":"tuple"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions[]","name":"permitted","type":"tuple[]"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitBatchTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails[]","name":"transferDetails","type":"tuple[]"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions","name":"permitted","type":"tuple"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails","name":"transferDetails","type":"tuple"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes32","name":"witness","type":"bytes32"},{"internalType":"string","name":"witnessTypeString","type":"string"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitWitnessTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions[]","name":"permitted","type":"tuple[]"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitBatchTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails[]","name":"transferDetails","type":"tuple[]"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes32","name":"witness","type":"bytes32"},{"internalType":"string","name":"witnessTypeString","type":"string"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitWitnessTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"address","name":"token","type":"address"}],"internalType":"struct IAllowanceTransfer.AllowanceTransferDetails[]","name":"transferDetails","type":"tuple[]"}],"name":"transferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"address","name":"token","type":"address"}],"name":"transferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

permit2_contract = w3.eth.contract(address=permit2_address, abi=abi_permit2)
universal_router_contract = w3.eth.contract(address=universal_router_address, abi=abi_universal_router)

# Test calls to Universal Router and Permit2 contracts
print((permit2_contract.functions.DOMAIN_SEPARATOR().call()).hex()) # print the DOMAIN_SEPARATOR of Permit2 contract, which is a hash including unique information about the contract and the chain, used in EIP-712 signatures
print(universal_router_contract.functions.supportsInterface('0x01ffc9a7').call())  # ERC165 interface ID

94c1dec87927751697bfc9ebf6fc4ca506bed30308b518f0e9d6c5f74bbafdb8
True


## Approving USTETH to the Permit2 contract

This step is similar to the ERC-20 `approve` function, but instead of approving Uniswap, we approve the Permit2 contract to spend our USTETH. This is necessary because when we execute the swap, the Universal Router will call Permit2 to transfer our USTETH.

In [99]:
# approve Permit2 to USTETH with max allowance

permit2_allowance = 2**256 - 1  # max

contract_function = USTETH_contract.functions.approve(
        permit2_address,
        permit2_allowance
)

trx_params = contract_function.build_transaction(
        {
            "from": wallet_public_address,
            "gas": 500_000,
            "maxPriorityFeePerGas": w3.eth.max_priority_fee,
            "maxFeePerGas": 100 * 10**9,
            "type": '0x2',
            "chainId": 11155111,
            "value": 0,
            "nonce": w3.eth.get_transaction_count(wallet_public_address),
        }
    )

signed_txn = w3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction

trx_hash = w3.eth.send_raw_transaction(raw_transaction)
print(f"Permit2 USTETH approve trx hash: {trx_hash.hex()}")

Permit2 USTETH approve trx hash: 685326f86661f2e43a17115f2de9d09fa4cd041774a6ad5aa7da9cc53e055c07


Checking the allowance we have just approved:

In [100]:
print(f'Permit2 allowance for USTETH: {USTETH_contract.functions.allowance(wallet_public_address, permit2_address).call()}')

Permit2 allowance for USTETH: 115792089237316195423570985008687907853269984665640564039457584007913129639935


## Approving USTETH to the Universal Router contract (Permit2 Internally)


In [101]:
import time

time.sleep(5)

# Parameters
spender = universal_router_address
amount = 2**160 - 1 # max uint160
expiration = int(time.time()) + (30 * 24 * 60 * 60) # example: 30 days from now

# approve(token, spender, amount, expiration)
permit2_function = permit2_contract.functions.approve(
    USTETH_token_address,
    spender,
    amount,
    expiration
)

trx_params_p2 = permit2_function.build_transaction({
    "from": wallet_public_address,
    "gas": 200_000,
    "maxPriorityFeePerGas": w3.eth.max_priority_fee,
    "maxFeePerGas": 100 * 10**9,
    "type": '0x2',
    "chainId": 11155111,
    "value": 0,
    "nonce": w3.eth.get_transaction_count(wallet_public_address),
})

print("Waiting for last transaction to be verified before sending the next one...")
w3.eth.wait_for_transaction_receipt(trx_hash)
print("Last transaction verified. Sending the next one...")

signed_txn_p2 = w3.eth.account.sign_transaction(trx_params_p2, wallet_private_key)
raw_transaction_p2 = signed_txn_p2.rawTransaction if hasattr(signed_txn_p2, 'rawTransaction') else signed_txn_p2.raw_transaction
trx_hash_p2 = w3.eth.send_raw_transaction(raw_transaction_p2)

print(f"Permit2 internal approve (USTETH) trx hash: {trx_hash_p2.hex()}")

Waiting for last transaction to be verified before sending the next one...
Last transaction verified. Sending the next one...
Permit2 internal approve (USTETH) trx hash: c757df13135349573fce6316107d653bb39d9d17b406f41f5a4b0db6ccc1250d


### Checking the current Permit2 nonce, allowance, and expiration
For security purposes (prevent Double Spending), the message you need to sign contains a unique `nonce`. It is incremented for each permit message you sign. The nonce depends on your account address and on the token and universal router addresses.

To know the current Permit2 nonce, allowance, and expiration:

In [103]:
p2_amount, p2_expiration, p2_nonce = permit2_contract.functions.allowance(
        wallet_public_address,
        USTETH_token_address,
        universal_router_address
).call()

print(f'Permit2 allowance amount: {p2_amount}')
print(f'Permit2 allowance expiration: {p2_expiration}')
print(f'Permit2 allowance nonce: {p2_nonce}')


Permit2 allowance amount: 1461501637330902918203684832716283019655932542975
Permit2 allowance expiration: 1774955475
Permit2 allowance nonce: 0


### Swap with Universal Router and Permit2

We now form swap transaction using universal router and v3_swap_exact_in:

First, we need to install Uniswap Universal Router Decoder & Encoder for encode and decode transaction params: 
```
pip install uniswap-universal-router-decoder
```

Next, we encode the transaction input data. The input data is the part of the transaction that will be executed by the UR smart contract, resulting in the actual swap. We’ll use the UR codec as follow:

*Hint: If your `eth-account` package version is 0.10.0 or lower, you may upgrade it to 0.10.1 or higher to avoid compatibility issues with the UR codec. You can do this by running:*

`pip install --upgrade eth-account`

In [104]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec, AllowanceTransferDetails
import time

deadline = int(time.time()) + (20 * 60) 

amount_in = int(1 * 10**18) # swap 10 USTETH into USTUSD, in Wei (18 decimals for USTETH)
min_amount_out = 0

codec = RouterCodec() # initialize the codec, which will be used to encode the function calls to the Universal Router

encoded_data = codec.encode.chain().v3_swap_exact_in( # chain() serilizes the function calls; v3_swap_exact_in() encodes a Uniswap V3 swap with exact input amount, which is the function we want to call in this example
        FunctionRecipient.SENDER,  # reciver
        amount_in,  # swap amount in, in Wei
        min_amount_out,  # Slippage Protection: minimum amount out, in Wei. If below this, the transaction will revert
        [   
            USTETH_token_address, # swap-from token address ()
            100, # fee tier for Uniswap V3 pool (100 = 0.01%)
            USTUSD_token_address,  # swap-into token address, which is the token we want to receive
        ],
    ).build(deadline)  # unix timestamp after which the trx will not be valid any more (to prevent the melicious use of the transaction if it gets stuck in the mempool for a long time)

print(f"Encoded data for Universal Router execute function: {encoded_data}")

Encoded data for Universal Router execute function: 0x3593564c000000000000000000000000000000000000000000000000000000000000006000000000000000000000000000000000000000000000000000000000000000a00000000000000000000000000000000000000000000000000000000069a423950000000000000000000000000000000000000000000000000000000000000001000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000de0b6b3a7640000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000a00000000000000000000000000000000000000000000000000000000000000001000000000000000000000000000000000000000000000000000000000000002b4e22e9951770b98d4f3b2ba6647f0f40a15a405700

Some explanations
Let’s breakdown the command:


* encode : tells the codec we want to encode an input data (as opposed to decode).

* chain() : the UR supports several commands chained in a single transaction. chain() initialises the chaining for one or more sub-commands.

* v3_swap_exact_in() : Instruct the router to use a V3 pool with known input amount (1 ueth in this case).

* FunctionRecipient.SENDER : the transaction’s sender will receive the output

* min_amount_out: If the swap results in less that this amount of uhkd, the transaction will be reverted.

* codec.get_default_deadline() : the timestamp after which the transaction will not be valid anymore.

* build() : This method build and encode the transaction input data.

#### Execute the transaction (Swap: USTETH -> USTUSD)

Finally, we can execute the transaction by sending it to the Ethereum network. We will sign the transaction with our wallet’s private key and then send it.

In [105]:
trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": w3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": w3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

signed_txn = w3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
txn_hash = w3.eth.send_raw_transaction(raw_transaction)
print("Hash of universal router swap transaction : ", w3.to_hex(txn_hash))

Hash of universal router swap transaction :  0xf76b21339f33796b33bced398a870edf69cd82665589cc86d2f8922f96ec8f5f


- Check the changes in token balances on metamask

## Atomic swap transaction (e.g, USTETH -> USTUSD and USTUSD -> USTETH at the same time)

Please note that beore doing atomic swap, you may need to repeat the approve steps including "Approve USTUSD to permit2 and Universal Router contract". 

Similarly, approving USTUSD token to Permit2 contract and the Universal Router contract at once 

In [106]:
import time

# approve Permit2 to spend USTUSD with max allowance
permit2_allowance = 2**256 - 1  # max

contract_function = USTUSD_contract.functions.approve(
        permit2_address,
        permit2_allowance
)

trx_params = contract_function.build_transaction(
        {
            "from": wallet_public_address,
            "gas": 500_000,
            "maxPriorityFeePerGas": w3.eth.max_priority_fee,
            "maxFeePerGas": 100 * 10**9,
            "type": '0x2',
            "chainId": 11155111,
            "value": 0,
            "nonce": w3.eth.get_transaction_count(wallet_public_address),
        }
    )
raw_transaction = w3.eth.account.sign_transaction(trx_params, wallet_private_key).raw_transaction
trx_hash = w3.eth.send_raw_transaction(raw_transaction)
print(f"Permit2 USTUSD approve trx hash: {trx_hash.hex()}")


# ==============================================================================================================================

# approve Universal Router to spend USTUSD with max allowance

time.sleep(5)

# Parameters
spender = universal_router_address
amount = 2**160 - 1 # max uint160
expiration = int(time.time()) + (30 * 24 * 60 * 60) # example: 30 days from now

# approve(token, spender, amount, expiration)
permit2_function = permit2_contract.functions.approve(
    USTUSD_token_address,
    spender,
    amount,
    expiration
)

trx_params_p2 = permit2_function.build_transaction({
    "from": wallet_public_address,
    "gas": 200_000,
    "maxPriorityFeePerGas": w3.eth.max_priority_fee,
    "maxFeePerGas": 100 * 10**9,
    "type": '0x2',
    "chainId": 11155111,
    "value": 0,
    "nonce": w3.eth.get_transaction_count(wallet_public_address),
})

print("Waiting for last transaction to be verified before sending the next one...")
w3.eth.wait_for_transaction_receipt(trx_hash)
print("Last transaction verified. Sending the next one...")

signed_txn_p2 = w3.eth.account.sign_transaction(trx_params_p2, wallet_private_key)
raw_transaction_p2 = signed_txn_p2.rawTransaction if hasattr(signed_txn_p2, 'rawTransaction') else signed_txn_p2.raw_transaction
trx_hash_p2 = w3.eth.send_raw_transaction(raw_transaction_p2)

print(f"Permit2 internal approve (USTUSD) trx hash: {trx_hash_p2.hex()}")

Permit2 USTUSD approve trx hash: e15008c6f53fc9c5bd2a36fb6926b275bfab6b20b143ddd80c79290d84f05a03
Waiting for last transaction to be verified before sending the next one...
Last transaction verified. Sending the next one...
Permit2 internal approve (USTUSD) trx hash: 6d07beb9ff4c7868ea296a44532fef00e2622d957e8722029b5d829c01e6cf91


In [107]:
print(f'Permit2 allowance for USTUSD: {USTUSD_contract.functions.allowance(wallet_public_address, permit2_address).call()}')
print(f'Permit2 allowance for USTUSD to Universal Router: {permit2_contract.functions.allowance(wallet_public_address, USTUSD_token_address, universal_router_address).call()}')

Permit2 allowance for USTUSD: 115792089237316195423570985008687907853269984665640564039457584007913129639935
Permit2 allowance for USTUSD to Universal Router: [1461501637330902918203684832716283019655932542975, 1774955055, 0]


### Swap both (USTETH -> USTUSD and USTUSD -> USTETH) at the same time

In [108]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
amount_in = int(1 * 10**18)
min_amount_out = 0

codec = RouterCodec()
encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # the output tokens are sent to the transaction sender
        amount_in,  # in Wei
        min_amount_out,  # in Wei
        [
            USTETH_token_address,  # checksum address of the token sent to the UR 
            100,
            USTUSD_token_address,  # checksum address of the received token
        ],
    ).v3_swap_exact_in(
        FunctionRecipient.SENDER,  # the output tokens are sent to the transaction sender
        amount_in,  # in Wei
        min_amount_out,  # in Wei
        [
            USTUSD_token_address,  # checksum address of the token sent to the UR 
            100,
            USTETH_token_address,  # checksum address of the received token
        ],
    ).build(codec.get_default_deadline())  # unix timestamp after which the trx will not be valid any more


# you can now sign and send the transaction to the UR
trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": w3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": w3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

signed_txn = w3.eth.account.sign_transaction(trx_params, wallet_private_key)
txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)
print("Hash of swap transaction : ", w3.to_hex(txn_hash))

Hash of swap transaction :  0xf5a388a4485b47f7764bfb3b9215f35c53ccab7dc72e94cd3a2d7a1f9df22573
